# IEEE-CIS Fraud Detection - Team 1 Member 2 Colab Setup

This notebook prepares a Colab-friendly preprocessing pipeline for **Team 1 / Member 2**:

- Download IEEE-CIS data with the Kaggle API
- Load transaction tables exactly once
- Downcast memory usage immediately after load
- Focus on transaction numericals, especially `V1-V339`
- Add null-handling and missingness-driven features
- Build optional PCA features over the V-block
- Export reusable `.parquet` outputs to Google Drive


## 1. Runtime setup

Before running the notebook:

1. In Colab, set runtime to **GPU**.
2. Prepare your Kaggle API token file: `kaggle.json`.
3. Run cells from top to bottom.


In [1]:
from pathlib import Path

RANDOM_SEED = 42
TARGET_COLUMN = "isFraud"
KAGGLE_DATASET = "ieee-fraud-detection"
PCA_COMPONENTS = 30
V_COLUMN_PREFIX = "V"

# Edit only this section when adapting the notebook.
PROJECT_ROOT = Path("/content/drive/MyDrive/ieee_cis_fraud/team1_member2")
RAW_DATA_DIR = PROJECT_ROOT / "raw"
INTERIM_DATA_DIR = PROJECT_ROOT / "interim"

OUTPUT_TRAIN_FILE = INTERIM_DATA_DIR / "train_member2_features.parquet"
OUTPUT_TEST_FILE = INTERIM_DATA_DIR / "test_member2_features.parquet"
FEATURE_MANIFEST_FILE = INTERIM_DATA_DIR / "feature_manifest.json"

PROJECT_ROOT, RAW_DATA_DIR, INTERIM_DATA_DIR

(PosixPath('/content/drive/MyDrive/ieee_cis_fraud/team1_member2'),
 PosixPath('/content/drive/MyDrive/ieee_cis_fraud/team1_member2/raw'),
 PosixPath('/content/drive/MyDrive/ieee_cis_fraud/team1_member2/interim'))

In [2]:
!pip -q install kaggle polars pyarrow fastparquet scikit-learn xgboost lightgbm catboost optuna shap seaborn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 37.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 33.1 MB/s eta 0:00:00


In [3]:
from google.colab import drive

drive.mount('/content/drive')

RAW_DATA_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DATA_DIR.mkdir(parents=True, exist_ok=True)

print(f"PROJECT_ROOT: {PROJECT_ROOT}")
print(f"RAW_DATA_DIR: {RAW_DATA_DIR}")
print(f"INTERIM_DATA_DIR: {INTERIM_DATA_DIR}")

Mounted at /content/drive
PROJECT_ROOT: /content/drive/MyDrive/ieee_cis_fraud/team1_member2
RAW_DATA_DIR: /content/drive/MyDrive/ieee_cis_fraud/team1_member2/raw
INTERIM_DATA_DIR: /content/drive/MyDrive/ieee_cis_fraud/team1_member2/interim


## 2. Kaggle API bootstrap

Run the next cell, upload your `kaggle.json`, and the notebook will place it in the expected Kaggle location.

In [4]:
import os
from google.colab import files

uploaded = files.upload()
assert 'kaggle.json' in uploaded, 'Please upload kaggle.json to continue.'

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'wb') as f:
    f.write(uploaded['kaggle.json'])

!chmod 600 /root/.kaggle/kaggle.json
print('Kaggle credentials configured.')

Saving kaggle.json to kaggle.json
Kaggle credentials configured.


In [5]:
import zipfile
from pathlib import Path

dataset_slug = KAGGLE_DATASET
download_dir = Path('/content/ieee_download')
download_dir.mkdir(parents=True, exist_ok=True)

!kaggle competitions download -c {dataset_slug} -p /content/ieee_download

zip_path = download_dir / f"{dataset_slug}.zip"
assert zip_path.exists(), f"Expected archive not found: {zip_path}"

with zipfile.ZipFile(zip_path, 'r') as zf:
    zf.extractall(RAW_DATA_DIR)

expected_files = ['train_transaction.csv', 'test_transaction.csv']
missing = [name for name in expected_files if not (RAW_DATA_DIR / name).exists()]
assert not missing, f"Missing expected files after unzip: {missing}"

print('Download and extraction complete.')
for name in expected_files:
    print('-', RAW_DATA_DIR / name)

100% 118M/118M [00:01<00:00, 85.8MB/s]

Download and extraction complete.
- /content/drive/MyDrive/ieee_cis_fraud/team1_member2/raw/train_transaction.csv
- /content/drive/MyDrive/ieee_cis_fraud/team1_member2/raw/test_transaction.csv


## 3. Feature engineering strategy

This notebook intentionally focuses on **Member 2 scope**:

- Transaction numeric columns only
- Explicit handling for `V1-V339`
- Missingness profiling and null-safe imputation
- Optional PCA compression for the V-block
- Parquet outputs for downstream teams


In [6]:
import gc
import json

import numpy as np
import pandas as pd
import polars as pl
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler


def memory_usage_mb(df: pd.DataFrame) -> float:
    return df.memory_usage(deep=True).sum() / 1024 ** 2


def downcast_numeric_frame(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()
    int_cols = result.select_dtypes(include=['int', 'int64', 'int32', 'int16']).columns
    float_cols = result.select_dtypes(include=['float', 'float64', 'float32']).columns

    for col in int_cols:
        result[col] = pd.to_numeric(result[col], downcast='integer')

    for col in float_cols:
        result[col] = pd.to_numeric(result[col], downcast='float')

    return result


def read_csv_once(path: Path, name: str) -> pd.DataFrame:
    df = pd.read_csv(path)
    before = memory_usage_mb(df)
    df = downcast_numeric_frame(df)
    after = memory_usage_mb(df)
    print(f"{name}: {before:.2f} MB -> {after:.2f} MB")
    return df


def get_v_columns(df: pd.DataFrame) -> list[str]:
    return [col for col in df.columns if col.startswith(V_COLUMN_PREFIX) and col[1:].isdigit()]


def build_missingness_features(df: pd.DataFrame, v_columns: list[str]) -> pd.DataFrame:
    result = df.copy()
    result['member2_total_null_count'] = result.isna().sum(axis=1).astype('int16')
    result['member2_v_null_count'] = result[v_columns].isna().sum(axis=1).astype('int16')
    result['member2_v_null_ratio'] = (result['member2_v_null_count'] / len(v_columns)).astype('float32')
    return result


def add_v_presence_flags(df: pd.DataFrame, selected_v_columns: list[str]) -> pd.DataFrame:
    result = df.copy()
    for col in selected_v_columns:
        result[f'{col}_was_missing'] = result[col].isna().astype('int8')
    return result


def add_transaction_numeric_features(df: pd.DataFrame) -> pd.DataFrame:
    result = df.copy()

    if 'TransactionAmt' in result.columns:
        amt = result['TransactionAmt'].astype('float32')
        result['member2_transactionamt_is_missing'] = amt.isna().astype('int8')
        result['member2_transactionamt_log1p'] = np.log1p(amt.fillna(0)).astype('float32')

    if 'dist1' in result.columns:
        result['member2_dist1_is_missing'] = result['dist1'].isna().astype('int8')
    if 'dist2' in result.columns:
        result['member2_dist2_is_missing'] = result['dist2'].isna().astype('int8')

    return result


def median_impute_numeric(train_df: pd.DataFrame, test_df: pd.DataFrame, columns: list[str]) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, float]]:
    imputer = SimpleImputer(strategy='median')
    train_values = imputer.fit_transform(train_df[columns])
    test_values = imputer.transform(test_df[columns])

    train_result = train_df.copy()
    test_result = test_df.copy()

    train_result.loc[:, columns] = train_values.astype('float32')
    test_result.loc[:, columns] = test_values.astype('float32')

    medians = {col: float(value) for col, value in zip(columns, imputer.statistics_)}
    return train_result, test_result, medians


def build_pca_features(train_df: pd.DataFrame, test_df: pd.DataFrame, source_columns: list[str], n_components: int) -> tuple[pd.DataFrame, pd.DataFrame, list[str], list[float]]:
    scaler = StandardScaler()
    pca = PCA(n_components=n_components, random_state=RANDOM_SEED)

    train_scaled = scaler.fit_transform(train_df[source_columns])
    test_scaled = scaler.transform(test_df[source_columns])

    train_pca = pca.fit_transform(train_scaled)
    test_pca = pca.transform(test_scaled)

    pca_columns = [f'member2_v_pca_{i + 1:02d}' for i in range(n_components)]
    train_pca_df = pd.DataFrame(train_pca, columns=pca_columns, index=train_df.index).astype('float32')
    test_pca_df = pd.DataFrame(test_pca, columns=pca_columns, index=test_df.index).astype('float32')

    return train_pca_df, test_pca_df, pca_columns, pca.explained_variance_ratio_.tolist()


In [7]:
train_transaction = read_csv_once(RAW_DATA_DIR / 'train_transaction.csv', 'train_transaction')
test_transaction = read_csv_once(RAW_DATA_DIR / 'test_transaction.csv', 'test_transaction')

print('train_transaction shape:', train_transaction.shape)
print('test_transaction shape:', test_transaction.shape)
print('train memory:', f'{memory_usage_mb(train_transaction):.2f} MB')
print('test memory:', f'{memory_usage_mb(test_transaction):.2f} MB')

train_transaction: 2062.07 MB -> 1203.22 MB
test_transaction: 1771.84 MB -> 1038.31 MB
train_transaction shape: (590540, 394)
test_transaction shape: (506691, 393)
train memory: 1203.22 MB
test memory: 1038.31 MB


In [8]:
v_columns = get_v_columns(train_transaction)
assert len(v_columns) == 339, f'Expected 339 V columns, found {len(v_columns)}'

base_keep_columns = ['TransactionID', TARGET_COLUMN, 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2']
base_keep_columns = [col for col in base_keep_columns if col in train_transaction.columns]
test_keep_columns = [col for col in base_keep_columns if col != TARGET_COLUMN]

selected_v_missing_flags = ['V1', 'V12', 'V35', 'V53', 'V86', 'V95', 'V130', 'V169', 'V201', 'V258', 'V307', 'V339']
selected_v_missing_flags = [col for col in selected_v_missing_flags if col in v_columns]

print('Base keep columns:', base_keep_columns)
print('Number of V columns:', len(v_columns))
print('Missingness flag columns:', selected_v_missing_flags)

Base keep columns: ['TransactionID', 'isFraud', 'TransactionAmt', 'ProductCD', 'card1', 'card2', 'card3', 'card5', 'addr1', 'addr2', 'dist1', 'dist2']
Number of V columns: 339
Missingness flag columns: ['V1', 'V12', 'V35', 'V53', 'V86', 'V95', 'V130', 'V169', 'V201', 'V258', 'V307', 'V339']


In [9]:
train_member2 = train_transaction[base_keep_columns + v_columns].copy()
test_member2 = test_transaction[test_keep_columns + v_columns].copy()

train_member2 = build_missingness_features(train_member2, v_columns)
test_member2 = build_missingness_features(test_member2, v_columns)

train_member2 = add_v_presence_flags(train_member2, selected_v_missing_flags)
test_member2 = add_v_presence_flags(test_member2, selected_v_missing_flags)

train_member2 = add_transaction_numeric_features(train_member2)
test_member2 = add_transaction_numeric_features(test_member2)

print('Feature engineering complete.')
print('train_member2 shape:', train_member2.shape)
print('test_member2 shape:', test_member2.shape)

Feature engineering complete.
train_member2 shape: (590540, 370)
test_member2 shape: (506691, 369)


In [10]:
train_member2, test_member2, v_medians = median_impute_numeric(train_member2, test_member2, v_columns)

numeric_non_v_columns = [
    col for col in train_member2.columns
    if col not in {TARGET_COLUMN, 'TransactionID'} and col not in v_columns and pd.api.types.is_numeric_dtype(train_member2[col])
]
train_member2, test_member2, extra_numeric_medians = median_impute_numeric(train_member2, test_member2, numeric_non_v_columns)

train_member2 = downcast_numeric_frame(train_member2)
test_member2 = downcast_numeric_frame(test_member2)

print('Median imputation complete.')
print('Imputed V columns:', len(v_medians))
print('Imputed non-V numeric columns:', len(extra_numeric_medians))

Median imputation complete.
Imputed V columns: 339
Imputed non-V numeric columns: 28


In [11]:
train_pca_df, test_pca_df, pca_columns, explained_variance_ratio = build_pca_features(
    train_member2,
    test_member2,
    v_columns,
    PCA_COMPONENTS,
)

train_member2 = pd.concat([train_member2, train_pca_df], axis=1)
test_member2 = pd.concat([test_member2, test_pca_df], axis=1)

del train_pca_df, test_pca_df
gc.collect()

print('PCA complete.')
print('PCA columns added:', pca_columns[:5], '...')
print('Explained variance (first 5):', [round(x, 6) for x in explained_variance_ratio[:5]])
print('Total explained variance:', round(sum(explained_variance_ratio), 6))

PCA complete.
PCA columns added: ['member2_v_pca_01', 'member2_v_pca_02', 'member2_v_pca_03', 'member2_v_pca_04', 'member2_v_pca_05'] ...
Explained variance (first 5): [0.180165, 0.10384, 0.058889, 0.040852, 0.033111]
Total explained variance: 0.79488


In [12]:
preview_cols = ['TransactionAmt'] if 'TransactionAmt' in train_member2.columns else []
preview_cols += ['member2_total_null_count', 'member2_v_null_count', 'member2_v_null_ratio']
train_member2_pl = pl.from_pandas(train_member2[['TransactionID'] + preview_cols + [TARGET_COLUMN]])

print(train_member2_pl.select([
    pl.len().alias('rows'),
    pl.col('member2_total_null_count').mean().alias('avg_total_null_count'),
    pl.col('member2_v_null_ratio').mean().alias('avg_v_null_ratio')
]))

shape: (1, 3)
┌────────┬──────────────────────┬──────────────────┐
│ rows   ┆ avg_total_null_count ┆ avg_v_null_ratio │
│ ---    ┆ ---                  ┆ ---              │
│ u32    ┆ f64                  ┆ f32              │
╞════════╪══════════════════════╪══════════════════╡
│ 590540 ┆ 147.680735           ┆ 0.430385         │
└────────┴──────────────────────┴──────────────────┘


In [13]:
assert len(train_member2) == len(train_transaction), 'Train row count changed during preprocessing.'
assert len(test_member2) == len(test_transaction), 'Test row count changed during preprocessing.'
assert TARGET_COLUMN in train_member2.columns, 'Target column missing from train output.'
assert TARGET_COLUMN not in test_member2.columns, 'Target column should not exist in test output.'
assert train_member2['TransactionID'].is_unique, 'Duplicate TransactionID found in train output.'
assert test_member2['TransactionID'].is_unique, 'Duplicate TransactionID found in test output.'
assert not train_member2[v_columns].isna().any().any(), 'V columns still contain nulls in train.'
assert not test_member2[v_columns].isna().any().any(), 'V columns still contain nulls in test.'

shared_columns = sorted(set(train_member2.columns) - {TARGET_COLUMN})
test_columns = sorted(test_member2.columns)
assert shared_columns == test_columns, f'Inconsistent columns. Unique to train: {set(shared_columns) - set(test_columns)}. Unique to test: {set(test_columns) - set(shared_columns)}'

del train_transaction, test_transaction
gc.collect()

print('Row count, null, and schema checks passed.')

Row count, null, and schema checks passed.


In [14]:
train_member2 = train_member2.drop(columns=v_columns)
test_member2 = test_member2.drop(columns=v_columns)

train_member2.to_parquet(OUTPUT_TRAIN_FILE, index=False)
test_member2.to_parquet(OUTPUT_TEST_FILE, index=False)

feature_manifest = {
    'target_column': TARGET_COLUMN,
    'join_key': 'TransactionID',
    'train_output': str(OUTPUT_TRAIN_FILE),
    'test_output': str(OUTPUT_TEST_FILE),
    'v_columns': v_columns,
    'selected_v_missing_flags': selected_v_missing_flags,
    'pca_components': PCA_COMPONENTS,
    'pca_columns': pca_columns,
    'pca_explained_variance_ratio': explained_variance_ratio,
    'v_medians_sample': dict(list(v_medians.items())[:10]),
    'extra_numeric_medians': extra_numeric_medians,
    'train_columns': train_member2.columns.tolist(),
    'test_columns': test_member2.columns.tolist(),
    'train_shape': list(train_member2.shape),
    'test_shape': list(test_member2.shape),
}

with open(FEATURE_MANIFEST_FILE, 'w', encoding='utf-8') as f:
    json.dump(feature_manifest, f, indent=2)

print('Exports written:')
print('-', OUTPUT_TRAIN_FILE)
print('-', OUTPUT_TEST_FILE)
print('-', FEATURE_MANIFEST_FILE)

Exports written:
- /content/drive/MyDrive/ieee_cis_fraud/team1_member2/interim/train_member2_features.parquet
- /content/drive/MyDrive/ieee_cis_fraud/team1_member2/interim/test_member2_features.parquet
- /content/drive/MyDrive/ieee_cis_fraud/team1_member2/interim/feature_manifest.json


In [15]:
train_reloaded = pd.read_parquet(OUTPUT_TRAIN_FILE)
test_reloaded = pd.read_parquet(OUTPUT_TEST_FILE)

assert train_reloaded.shape == train_member2.shape
assert test_reloaded.shape == test_member2.shape

sample_columns = ['TransactionID']
if 'TransactionAmt' in train_reloaded.columns:
    sample_columns.append('TransactionAmt')
sample_columns += ['member2_v_null_count'] + pca_columns[:3]
if TARGET_COLUMN in train_reloaded.columns:
    sample_columns.append(TARGET_COLUMN)

print('Parquet reload validation passed.')
print(train_reloaded[sample_columns].head())

Parquet reload validation passed.
   TransactionID  TransactionAmt  member2_v_null_count  member2_v_pca_01  \
0        2987000            68.5                   177         -1.015884   
1        2987001            29.0                   170         -0.948673   
2        2987002            59.0                   159         -1.027161   
3        2987003            50.0                   170          1.096706   
4        2987004            50.0                    94          0.774378   

   member2_v_pca_02  member2_v_pca_03  isFraud  
0         -2.243620          0.160115        0  
1         -0.875562         -0.321312        0  
2         -2.096239          0.163296        0  
3         -2.684554         -0.057901        0  
4         -2.097016         -2.628136        0  


## 4. Handoff notes

The generated parquet files are intended as reusable inputs for later teams.

- `train_member2_features.parquet` keeps `isFraud`
- `test_member2_features.parquet` excludes `isFraud`
- `feature_manifest.json` records the V-column strategy, PCA outputs, and exported schema

This notebook uses `V1-V339` only inside the pipeline (imputation, missingness features, and PCA). **Raw `V1-V339` columns are dropped before export**; downstream teams receive the V-block as **`member2_v_pca_*` components only**, together with the other base and engineered columns.


# Merge with member1

In [1]:
from google.colab import drive
drive.mount("/content/drive")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# a.py is src/features/merge_member1_member2.py
!python -u a.py \
--member1-train "/content/drive/MyDrive/ieee_cis_fraud/team1_member1/interim/train_member1_features.parquet" \
--member1-test "/content/drive/MyDrive/ieee_cis_fraud/team1_member1/interim/test_member1_features.parquet" \
--member2-train "/content/drive/MyDrive/ieee_cis_fraud/team1_member2/interim/train_member2_features.parquet" \
--member2-test "/content/drive/MyDrive/ieee_cis_fraud/team1_member2/interim/test_member2_features.parquet" \
--out-train "/content/drive/MyDrive/ieee_cis_fraud/merged/train_member1_member2_merged.parquet" \
--out-test "/content/drive/MyDrive/ieee_cis_fraud/merged/test_member1_member2_merged.parquet" \
--out-manifest "/content/drive/MyDrive/ieee_cis_fraud/merged/member1_member2_merge_manifest.json" \
--overlap-strategy prefer-member2


Reading parquet schemas...
Detected overlapping columns: 10
Using overlap strategy: prefer-member2
Loading Member 1 train...
Loaded Member 1 train: shape=(590540, 92)
Loading Member 1 test...
Loaded Member 1 test: shape=(506691, 91)
Loading Member 2 train...
Loaded Member 2 train: shape=(590540, 61)
Loading Member 2 test...
Loaded Member 2 test: shape=(506691, 60)
Merging train...
Merged train: shape=(590540, 152)
Merging test...
Merged test: shape=(506691, 150)
Writing merged train parquet...
Writing merged test parquet...
Writing manifest...
Merged outputs written:
- train: /content/drive/MyDrive/ieee_cis_fraud/merged/train_member1_member2_merged.parquet
- test: /content/drive/MyDrive/ieee_cis_fraud/merged/test_member1_member2_merged.parquet
- manifest: /content/drive/MyDrive/ieee_cis_fraud/merged/member1_member2_merge_manifest.json
- merged train shape: (590540, 151)
- merged test shape: (506691, 150)


In [9]:
import pandas as pd

train = pd.read_parquet("/content/drive/MyDrive/ieee_cis_fraud/merged/train_member1_member2_merged.parquet")
test = pd.read_parquet("/content/drive/MyDrive/ieee_cis_fraud/merged/test_member1_member2_merged.parquet")

print(train.shape)
print(test.shape)
print(train.columns[:20].tolist())

(590540, 151)
(506691, 150)
['TransactionID', 'isFraud', 'TransactionDT', 'card4', 'card6', 'P_emaildomain', 'R_emaildomain', 'C1', 'C2', 'C3', 'C4', 'C5', 'C6', 'C7', 'C8', 'C9', 'C10', 'C11', 'C12', 'C13']
